# 🧹 Dataset Cleaning Notebook
### ACLEDA Bank FAQ Dataset — Khmer Language

**Purpose:** Clean the raw `dataasilida_cleaned.csv` file and produce `cleaned_data.csv`  
**Input file:** `dataasilida_cleaned.csv` (upload this to Colab first)  
**Output file:** `cleaned_data.csv` (auto-saved, ready for Gemma 3 fine-tuning)

---

### What this notebook fixes:
| # | Problem | Fix Applied |
|---|---------|-------------|
| 1 | Header row is wrapped in quotes (`"input,output"`) making it look like 1 column | Detect and skip bad header |
| 2 | Some rows have question & answer merged inside a single quote block: `"Question? ,Answer"` | Split on first `?` delimiter |
| 3 | Invisible Khmer zero-width spaces `\u200b`, `\ufeff` from web scraping | Strip with regex |
| 4 | Duplicate question-answer pairs | Drop with pandas |
| 5 | Leftover quotes and escaped newlines `\\n` | Strip and normalize |


## Step 1: Connect Google Drive

Run the cell below — it will ask you to sign in. After that, your Drive files are accessible.


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# ── Set your folder path here ──────────────────────────────────────────────
# After mounting, your Drive files are under /content/drive/MyDrive/
# Change "TP5" below to match YOUR folder name on Google Drive
FOLDER = "/content/drive/MyDrive/TP5"

import os
print("Files in your Drive folder:")
for f in os.listdir(FOLDER):
    print(" ", f)


Mounted at /content/drive
Files in your Drive folder:
  dataasilida_cleaned.csv
  Gemma3_(4B).ipynb
  data_cleaning.ipynb


In [2]:
# Step 2: Inspect the raw file — see what is broken
with open(FOLDER + "/dataasilida_cleaned.csv", "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

print(f"Total lines in raw file: {len(raw_lines)}")
print()
print("=== First 3 lines (raw) ===")
for i, line in enumerate(raw_lines[:3]):
    print(f"Line {i}: {repr(line[:120])}")


Total lines in raw file: 522

=== First 3 lines (raw) ===
Line 0: '"input,output"\n'
Line 1: '"តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី","ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ ឬកាតអេឡិចត្រូនិក បោះផ្សា'
Line 2: '"តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ផ្តល់នូវអត្ថប្រយោជន៍អ្វីខ្លះ","លោកអ្នកទទួលបានអត្ថប្រយោជន៍ពីការប្រើប្រាស់ ប័ណ្ណឥណ'


In [3]:
# Step 3: Find and display the broken / malformed lines
import csv

print("=== Lines that did NOT parse into exactly 2 columns ===")
bad_count = 0
for idx, line in enumerate(raw_lines):
    stripped = line.strip()
    if not stripped:
        continue
    if idx == 0 and "input" in stripped and "output" in stripped:
        print(f"Line   0: [BAD HEADER] {repr(stripped[:80])}")
        bad_count += 1
        continue
    parsed = list(csv.reader([stripped]))[0]
    if len(parsed) != 2:
        print(f"Line {idx:3d}: [{len(parsed)} col] {repr(stripped[:100])}")
        bad_count += 1

print(f"\nTotal bad / malformed lines found: {bad_count}")


=== Lines that did NOT parse into exactly 2 columns ===
Line   0: [BAD HEADER] '"input,output"'
Line  97: [1 col] '"តើប័ណ្ណនិម្មិតអេស៊ីលីដា មានកំណត់ទំហំទឹកប្រាក់ និងចំនួនប័ណ្ណ ដែរ ឬទេ ? ,លោក-អ្នកអាចបង្កើតប័ណ្ណនិម្មិ'
Line  98: [1 col] '"តើប័ណ្ណនិម្មិតអេស៊ីលីដា មានសុពលភាពប្រើប្រាស់បានរហូតដល់រយៈពេលប៉ុន្មាន ? ,ប័ណ្ណនិម្មិតអេស៊ីលីដា មានសុ'
Line 100: [1 col] '"តើការបង្កើតប័ណ្ណនិម្មិតអេស៊ីលីដា មានគិតថ្លៃសេវា ដែរ ឬទេ ? ,ការបង្កើតប័ណ្ណនិម្មិតអេស៊ីលីដា គឺ មិនមាន'
Line 392: [1 col] '"តើសាធារណជនអាចប្តូរប្រាក់នៅឯធនាគារជាតិនៃកម្ពុជាបានឬទេ ? ,អ្នកអាចប្តូរប្រាក់រៀល ឬកាសនៅធនាគារជាតិនៃកម្'
Line 394: [1 col] '"ហេតុអ្វីបានជាប្រាក់រៀលមានសារៈសំខាន់ចំពោះសេដ្ឋកិច្ចកម្ពុជា? ,ការប្រើប្រាស់ប្រាក់រៀលគឺពិតជាមានសារៈសំខ'
Line 418: [1 col] 'តើឯកសារតម្រូវដើម្បីស្នើសុំកម្ចីសាច់ប្រាក់មានអ្វីខ្លះ ? \\n – មាន\u200bអត្តសញ្ញាណ\u200bប័ណ្ណ ឬ លិខិត\u200bឆ្លងដែន\u200bច្ប'
Line 423: [1 col] '"ប្រសិនបើខ្ញុំបង់ប្រាក់យឺត តើខ្ញុំត្រូវបង់ប្រាក់ពិន័យប៉ុន្មាន ? ,– ប្រសិនបើ\u200bបង់ចន្លោះ\u200bពីថ្ងៃទី ៦-១០ '

Total bad / malformed lin

In [4]:
# Step 4: Show rows that contain invisible zero-width space characters (\u200b)
import re

zwsp_rows = []
for idx, line in enumerate(raw_lines[1:], start=1):  # skip header
    if "\u200b" in repr(line) or "\ufeff" in repr(line):
        zwsp_rows.append(idx)

print(f"Rows containing invisible zero-width spaces (\\u200b / \\ufeff): {len(zwsp_rows)}")
print("Row indices:", zwsp_rows[:20], "..." if len(zwsp_rows) > 20 else "")


Rows containing invisible zero-width spaces (\u200b / \ufeff): 0
Row indices: [] 


In [5]:
# Step 5: Run the full cleaning pipeline
import csv
import re
import pandas as pd

raw_file    = FOLDER + "/dataasilida_cleaned.csv"   # original raw file (UNCHANGED)
output_file = FOLDER + "/cleaned_data.csv"           # output of this notebook

with open(raw_file, "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

cleaned_rows = []

for idx, line in enumerate(raw_lines):
    line = line.strip()

    # Skip empty lines
    if not line:
        continue

    # Fix 1: Skip bad header "input,output" (whole thing is in quotes = 1 column)
    if idx == 0 and ("input" in line and "output" in line):
        continue

    # Fix 2: Parse with CSV reader
    parsed = list(csv.reader([line]))[0]

    if len(parsed) == 2:
        # Normal row
        q, a = parsed[0], parsed[1]

    elif len(parsed) == 1:
        # Broken row: question & answer merged inside one quote block
        # e.g.  "What is this ? ,This is the answer"
        text = parsed[0]
        if "?" in text:
            parts = text.split("?", 1)   # split on the FIRST question mark
            q = parts[0] + "?"
            a = parts[1].strip()
            if a.startswith(","):
                a = a[1:].strip()        # remove leading comma after the ?
        else:
            continue   # can't recover, skip
    else:
        continue

    # Fix 3: Remove invisible unicode characters common in scraped Khmer text
    #   \u200b = zero-width space
    #   \ufeff = BOM (byte-order mark)
    #   \u200e / \u200f = left-to-right / right-to-left marks
    q = re.sub(r"[\u200b\ufeff\u200e\u200f]", "", q).strip()
    a = re.sub(r"[\u200b\ufeff\u200e\u200f]", "", a).strip()

    # Fix 4: Strip leftover surrounding quotes
    q = q.strip('"').strip("'")
    a = a.strip('"').strip("'")

    # Fix 5: Normalize escaped newlines (\n written as literal text -> real newline)
    a = a.replace("\\n", "\n").strip()

    # Only keep rows that have both a question and an answer
    if q and a:
        cleaned_rows.append({"input": q, "output": a})

# Fix 6: Remove duplicate rows
df = pd.DataFrame(cleaned_rows)
before_dedup = len(df)
df = df.drop_duplicates(subset=["input", "output"]).reset_index(drop=True)
duplicates_removed = before_dedup - len(df)

# Save as cleaned_data.csv with proper header and quoting
df.to_csv(output_file, index=False, quoting=csv.QUOTE_MINIMAL, encoding="utf-8")

print("=" * 50)
print("       CLEANING COMPLETE ✅")
print("=" * 50)
print(f"  Raw lines read     : {len(raw_lines)}")
print(f"  Rows parsed OK     : {before_dedup}")
print(f"  Duplicates removed : {duplicates_removed}")
print(f"  Final clean rows   : {len(df)}")
print(f"  Null values        : {df.isnull().sum().sum()}")
print(f"  Saved to           : {output_file}")
print("=" * 50)


       CLEANING COMPLETE ✅
  Raw lines read     : 522
  Rows parsed OK     : 521
  Duplicates removed : 3
  Final clean rows   : 518
  Null values        : 0
  Saved to           : /content/drive/MyDrive/TP5/cleaned_data.csv


In [6]:
# Step 6: Validate the cleaned output
import pandas as pd

df_check = pd.read_csv(FOLDER + "/cleaned_data.csv")

print("Shape:", df_check.shape)
print("Columns:", df_check.columns.tolist())
print("Null values:\n", df_check.isnull().sum())
print("\nSample rows:")
df_check.head(5)


Shape: (518, 2)
Columns: ['input', 'output']
Null values:
 input     0
output    0
dtype: int64

Sample rows:


,input,output
0,តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី,ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ...
1,តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ផ្តល់នូ...,លោកអ្នកទទួលបានអត្ថប្រយោជន៍ពីការប្រើប្រាស់ ប័ណ្...
2,តើធ្វើដូចម្តេចទើបខ្ញុំអាច ស្នើសុំប័ណ្ណឥណពន្ធចេ...,លោកអ្នកអាចស្នើសុំ ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដ...
3,តើខ្ញុំអាចប្រើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ដក...,លោកអ្នកអាចដកប្រាក់បានរហូតដល់ ១០០០០ដុល្លារអាមេរ...
4,តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា មានសុពល...,ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា មានសុពលភា...


In [7]:
# Step 7: Preview a few rows — verify input/output look correct
for i, row in df_check.head(3).iterrows():
    print(f"--- Row {i+1} ---")
    print("Q:", row['input'])
    print("A:", row['output'][:150])
    print()


--- Row 1 ---
Q: តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី
A: ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ ឬកាតអេឡិចត្រូនិក បោះផ្សាយដោយ ធនាគារ អេស៊ីលីដា ភីអិលស៊ី សម្រាប់ផ្តល់ជូនអតិថិជនក្នុងការធ្វើប្រតិបត្តិការត

--- Row 2 ---
Q: តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ផ្តល់នូវអត្ថប្រយោជន៍អ្វីខ្លះ
A: លោកអ្នកទទួលបានអត្ថប្រយោជន៍ពីការប្រើប្រាស់ ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ដូចខាងក្រោម៖ -អាចដកសាច់ប្រាក់បានឆាប់រហ័សពីគ្រប់ម៉ាស៊ីន អេធីអឹម ទាំងអស់ដ

--- Row 3 ---
Q: តើធ្វើដូចម្តេចទើបខ្ញុំអាច ស្នើសុំប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា បាន
A: លោកអ្នកអាចស្នើសុំ ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា បានដោយគ្រាន់តែ បើកគណនីមួយនៅធនាគារ អេស៊ីលីដា សាខាណាមួយដែលនៅជិតលោកអ្នកបំផុត ហើយគណនីនោះអាចជា គណនីប



## ✅ Done!

`cleaned_data.csv` is now saved in your Colab session.

**Next step:** Open `Gemma3_(4B).ipynb` and upload both:
- `dataasilida_cleaned.csv` (raw, unchanged)
- `cleaned_data.csv` (the output of this notebook)

The Gemma notebook will load `cleaned_data.csv` automatically for training.
